In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

question = '인프런 강의 목록'

llm.invoke(question)

AIMessage(content='인프런은 다양한 온라인 강의를 제공하는 플랫폼으로, 주로 IT, 프로그래밍, 데이터 분석, 디자인 등 여러 분야의 강의를 포함하고 있습니다. 구체적인 강의 목록은 시간이 지남에 따라 업데이트되므로, 최신 정보를 확인하려면 인프런 공식 웹사이트를 방문하는 것이 좋습니다.\n\n일반적으로 인프런에서 제공하는 강의 카테고리는 다음과 같습니다:\n\n1. **프로그래밍**: Python, Java, JavaScript, C++, Ruby 등\n2. **웹 개발**: HTML, CSS, React, Vue.js, Node.js 등\n3. **모바일 개발**: Android, iOS, Flutter 등\n4. **데이터 분석**: 데이터 시각화, 머신러닝, R, SQL 등\n5. **디자인**: UI/UX 디자인, 포토샵, 일러스트레이터 등\n6. **DevOps**: Docker, Kubernetes, CI/CD 등\n7. **클라우드**: AWS, Azure, GCP 등\n\n각 카테고리 내에서 초급, 중급, 고급 수준의 강의를 선택할 수 있으며, 강의의 내용, 강사, 수강 후기 등을 참고하여 자신에게 맞는 강의를 찾을 수 있습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 270, 'prompt_tokens': 13, 'total_tokens': 283, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'im

In [9]:
from langchain_core.prompts import PromptTemplate

text_prompt = '아래 질문에 답변해주세요 \n\n {question}'

prompt_template = PromptTemplate(template=text_prompt, input_variables=["question"])
message = prompt_template.invoke({"question": question})


In [10]:
llm.invoke(message)

AIMessage(content='인프런은 다양한 주제의 온라인 강의를 제공하는 플랫폼입니다. 강의 목록은 주기적으로 업데이트되므로, 최신 강의 목록을 확인하려면 인프런 웹사이트를 직접 방문하는 것이 가장 좋습니다. 일반적으로 인프런에서는 다음과 같은 주제의 강의를 찾을 수 있습니다:\n\n1. **프로그래밍**: Python, Java, JavaScript, C++, Ruby 등\n2. **웹 개발**: HTML, CSS, React, Vue.js, Node.js 등\n3. **데이터 과학**: 데이터 분석, 머신러닝, 인공지능, R, SQL 등\n4. **모바일 개발**: Android, iOS, Flutter 등\n5. **디자인**: UI/UX 디자인, 그래픽 디자인, 포토샵 등\n6. **DevOps**: Docker, Kubernetes, CI/CD 등\n7. **클라우드**: AWS, Azure, GCP 등\n\n각 강의는 강사, 난이도, 수강 기간, 가격 등의 정보와 함께 제공됩니다. 특정 강의에 대한 정보가 필요하시다면, 인프런 웹사이트에서 검색해 보시기 바랍니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 249, 'prompt_tokens': 21, 'total_tokens': 270, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'mo

In [14]:
%pip install beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 2.2 MB/s eta 0:00:00 0:00:01

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader('https://www.inflearn.com/courses/Applied-ai')

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1500,
    chunk_overlap = 200,
)

document_list = loader.load_and_split(text_splitter=text_splitter)

In [17]:
from langchain_upstage import UpstageEmbeddings
from langchain_chroma import Chroma

embedding = UpstageEmbeddings(model='solar-embedding-1-large')

database = Chroma.from_documents(documents=document_list, embedding=embedding, collection_name='inflearn-test', persist_directory='./chroma_new')

retriver = database.as_retriever(search_kwargs={'k': 4})

retrieved_docs = retriver.invoke(question)

In [18]:
retrieved_docs

[Document(id='7523a1b6-eac3-4ebd-9f1f-9955d03c65af', metadata={'source': 'https://www.inflearn.com/courses/Applied-ai', 'title': '인프런 - 최대 AI, AX 강의 학습 플랫폼', 'language': 'ko', 'description': '프로그래밍, 인공지능, 데이터, 마케팅, 디자인등 입문부터 실전까지 업계 최고 선배들에게 배울 수 있는 곳.'}, page_content='인프런 - 최대 AI, AX 강의 학습 플랫폼기업 교육평생교육이용권AI 교육 90% 환급NN인프런 - 최대 AI, AX 강의 학습 플랫폼')]

In [19]:
rag_prompt_template = PromptTemplate(
    template='''
    You are a helpful assistant that can answer questions about the Ingflearn website.
    You are given the following context:
    {context}
    Qeustion:{question}
    ''',
    input_variables=['context', 'question']
)

rag_chain = rag_prompt_template | llm

message = rag_chain.invoke({"context": retrieved_docs, "question": question})

In [21]:
message

AIMessage(content='인프런에서는 프로그래밍, 인공지능, 데이터, 마케팅, 디자인 등 다양한 분야의 강의를 제공합니다. 강의는 입문부터 실전까지 다양한 수준으로 구성되어 있으며, 업계 최고 선배들에게 배울 수 있는 기회를 제공합니다. 구체적인 강의 목록은 인프런 웹사이트를 방문하여 확인하실 수 있습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 205, 'total_tokens': 282, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_be22d4f6f5', 'id': 'chatcmpl-EUAtmQizLdy2NcUYkGlqP7rVbMtJm', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0f799-b268-7ba2-bd3c-c0fa962b50f9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 205, 'output_tokens': 77, 'total_

In [22]:
from langchain_core.runnables import RunnablePassthrough

rag_chain2 = (
    {"context": retriver, "question": RunnablePassthrough()}
    | rag_prompt_template
    | llm
)

rag_chain2.invoke(question)

AIMessage(content='인프런에서는 프로그래밍, 인공지능, 데이터, 마케팅, 디자인 등 다양한 분야의 강의를 제공합니다. 강의는 입문부터 실전까지 다양한 수준으로 구성되어 있으며, 업계 최고 선배들에게 배울 수 있는 기회를 제공합니다. 강의 목록은 인프런 웹사이트에서 확인할 수 있습니다. 더 자세한 내용은 [인프런 웹사이트](https://www.inflearn.com/courses/Applied-ai)에서 확인해 보세요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 101, 'prompt_tokens': 204, 'total_tokens': 305, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_894e7baa82', 'id': 'chatcmpl-EUAyFBjNaGRa5terBAilGj3nzkjgR', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0f79d-eeca-70d0-a74a-7e9c8dac0bb8-0', tool_calls=[], invalid_tool_calls=[